# Prometheus + Grafana Dashboard for vLLM + GPU Metrics

Deploys standalone Prometheus and Grafana to monitor **vLLM inference** and **cluster GPU metrics** (NVIDIA DCGM).

| | |
|---|---|
| **Prometheus** | Scrapes vLLM `/metrics` + all `nvidia-dcgm-exporter` pods (every GPU node) |
| **Grafana** | Dashboard: GPU nodes, utilization, memory, temp, power + vLLM throughput/latency |
| **Access** | OpenShift Route (anonymous auth, no login needed) |

## Flow

1. Discover cluster nodes/GPUs via Kubernetes API
2. Deploy Prometheus (vLLM + DCGM scrape targets)
3. Deploy Grafana with pre-built dashboard
4. Open Grafana Route URL in browser
5. Run `grpo_vllm_training.ipynb` to see live GPU + vLLM metrics


## Imports and cluster config

In [1]:
import json
import os
import time

import requests
from kubernetes import client as k8s

In [2]:
SA_TOKEN_PATH = "/var/run/secrets/kubernetes.io/serviceaccount/token"
SA_NS_PATH = "/var/run/secrets/kubernetes.io/serviceaccount/namespace"

if not os.getenv("OPENSHIFT_API_URL") and os.path.exists(SA_TOKEN_PATH):
    os.environ["OPENSHIFT_API_URL"] = "https://kubernetes.default.svc"
    with open(SA_TOKEN_PATH) as f:
        os.environ["NOTEBOOK_USER_TOKEN"] = f.read()
if not os.getenv("NOTEBOOK_NAMESPACE") and os.path.exists(SA_NS_PATH):
    with open(SA_NS_PATH) as f:
        os.environ["NOTEBOOK_NAMESPACE"] = f.read().strip()

api_server = os.getenv("OPENSHIFT_API_URL")
token = os.getenv("NOTEBOOK_USER_TOKEN")
NAMESPACE = os.getenv("NOTEBOOK_NAMESPACE", "<your-namespace>")

VLLM_SERVICE = "grpo-vllm-rollout"
PROM_NAME = "vllm-prometheus"
GRAFANA_NAME = "vllm-grafana"

if not api_server or not token:
    raise RuntimeError("OPENSHIFT_API_URL and NOTEBOOK_USER_TOKEN must be set.")
if NAMESPACE == "<your-namespace>":
    raise RuntimeError("Set NOTEBOOK_NAMESPACE before running.")

configuration = k8s.Configuration()
configuration.host = api_server
configuration.verify_ssl = True
for candidate in [
    "/var/run/secrets/kubernetes.io/serviceaccount/ca.crt",
    os.getenv("SSL_CERT_FILE"),
    os.getenv("REQUESTS_CA_BUNDLE"),
]:
    if candidate and os.path.exists(candidate):
        configuration.ssl_ca_cert = candidate
        break
configuration.api_key = {"authorization": f"Bearer {token}"}
api_client = k8s.ApiClient(configuration)
apps = k8s.AppsV1Api(api_client)
core = k8s.CoreV1Api(api_client)

print(f"Namespace: {NAMESPACE}")
print(f"vLLM service target: {VLLM_SERVICE}.{NAMESPACE}.svc:8000")

Namespace: grpoxtrainer
vLLM service target: grpo-vllm-rollout.grpoxtrainer.svc:8000


## Deploy Prometheus

In [ ]:
import subprocess
import sys

# Discover cluster + GPU inventory (falls back if SA cannot list nodes)
cluster_nodes = gpu_nodes = total_gpus_alloc = 0
try:
    nodes = core.list_node().items
    cluster_nodes = len(nodes)
    for n in nodes:
        alloc = n.status.allocatable or {}
        gpu = alloc.get("nvidia.com/gpu")
        if gpu and int(gpu) > 0:
            gpu_nodes += 1
            total_gpus_alloc += int(gpu)
    print(f"Cluster: {cluster_nodes} nodes, {gpu_nodes} GPU nodes, {total_gpus_alloc} GPUs allocatable")
except Exception as e:
    print(f"Node list unavailable ({e}); GPU counts will come from DCGM scrape only.")

# Discover all NVIDIA DCGM exporter pod IPs (one per GPU node)
dcgm_pods = core.list_namespaced_pod(
    namespace="nvidia-gpu-operator",
    label_selector="app=nvidia-dcgm-exporter",
).items
dcgm_targets = []
for pod in dcgm_pods:
    if pod.status.pod_ip:
        dcgm_targets.append(f"{pod.status.pod_ip}:9400")
print(f"DCGM exporters: {len(dcgm_targets)} pods -> {dcgm_targets}")
if gpu_nodes == 0 and dcgm_targets:
    gpu_nodes = len(dcgm_targets)
    print(f"  Inferred {gpu_nodes} GPU nodes from DCGM exporter count")

dcgm_yaml = ""
if dcgm_targets:
    dcgm_yaml = "  - job_name: dcgm\n    metrics_path: /metrics\n    static_configs:\n      - targets:\n"
    for t in dcgm_targets:
        dcgm_yaml += f"          - {t}\n"
    dcgm_yaml += "        labels:\n          service: dcgm\n"

prometheus_config = f"""global:
  scrape_interval: 5s
  evaluation_interval: 5s

scrape_configs:
  - job_name: vllm
    metrics_path: /metrics
    static_configs:
      - targets:
          - {VLLM_SERVICE}.{NAMESPACE}.svc:8000
        labels:
          service: vllm
{dcgm_yaml}"""

prom_configmap = {
    "apiVersion": "v1",
    "kind": "ConfigMap",
    "metadata": {"name": f"{PROM_NAME}-config", "namespace": NAMESPACE},
    "data": {"prometheus.yml": prometheus_config},
}

prom_deployment = {
    "apiVersion": "apps/v1",
    "kind": "Deployment",
    "metadata": {"name": PROM_NAME, "namespace": NAMESPACE},
    "spec": {
        "replicas": 1,
        "selector": {"matchLabels": {"app": PROM_NAME}},
        "template": {
            "metadata": {"labels": {"app": PROM_NAME}},
            "spec": {
                "serviceAccountName": "grpoxtrainerwb1",
                "containers": [
                    {
                        "name": "prometheus",
                        "image": "prom/prometheus:latest",
                        "args": [
                            "--config.file=/etc/prometheus/prometheus.yml",
                            "--storage.tsdb.path=/prometheus",
                            "--storage.tsdb.retention.time=2h",
                            "--web.enable-lifecycle",
                        ],
                        "ports": [{"containerPort": 9090, "name": "http"}],
                        "resources": {
                            "requests": {"cpu": "200m", "memory": "256Mi"},
                            "limits": {"cpu": "500m", "memory": "512Mi"},
                        },
                        "volumeMounts": [
                            {"name": "config", "mountPath": "/etc/prometheus"},
                            {"name": "data", "mountPath": "/prometheus"},
                        ],
                    }
                ],
                "volumes": [
                    {
                        "name": "config",
                        "configMap": {"name": f"{PROM_NAME}-config"},
                    },
                    {"name": "data", "emptyDir": {}},
                ],
            },
        },
    },
}

prom_service = {
    "apiVersion": "v1",
    "kind": "Service",
    "metadata": {"name": PROM_NAME, "namespace": NAMESPACE},
    "spec": {
        "selector": {"app": PROM_NAME},
        "ports": [{"port": 9090, "targetPort": 9090, "name": "http"}],
    },
}

def _apply(kind, name, create_fn, patch_fn, body):
    try:
        create_fn(namespace=NAMESPACE, body=body)
        print(f"  Created {kind}/{name}")
    except Exception:
        patch_fn(name=name, namespace=NAMESPACE, body=body)
        print(f"  Updated {kind}/{name}")

_apply("ConfigMap", f"{PROM_NAME}-config", core.create_namespaced_config_map, core.patch_namespaced_config_map, prom_configmap)
_apply("Deployment", PROM_NAME, apps.create_namespaced_deployment, apps.patch_namespaced_deployment, prom_deployment)
_apply("Service", PROM_NAME, core.create_namespaced_service, core.patch_namespaced_service, prom_service)
print("Prometheus deployed (vLLM + DCGM GPU metrics).")


## Deploy Grafana with pre-built vLLM dashboard

In [ ]:
datasource_provisioning = f"""apiVersion: 1
datasources:
  - name: Prometheus
    type: prometheus
    access: proxy
    url: http://{PROM_NAME}.{NAMESPACE}.svc:9090
    isDefault: true
    editable: true
"""

dashboard_provisioning = """apiVersion: 1
providers:
  - name: default
    orgId: 1
    folder: ""
    type: file
    disableDeletion: false
    updateIntervalSeconds: 10
    options:
      path: /var/lib/grafana/dashboards
      foldersFromFilesStructure: false
"""

# Build dashboard with cluster GPU + vLLM panels
import importlib.util
_dash_path = "/opt/app-root/src/grafana_dashboard_vllm.py"
if not __import__("os").path.exists(_dash_path):
    _dash_path = __import__("os").path.join(
        __import__("os").path.dirname(__import__("os").path.abspath(".")),
        "scripts", "grafana_dashboard_vllm.py",
    )
_spec = importlib.util.spec_from_file_location("grafana_dashboard_vllm", _dash_path)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
dashboard_json = _mod.build_dashboard(
    cluster_nodes=cluster_nodes,
    gpu_nodes=gpu_nodes,
    total_gpus_alloc=total_gpus_alloc,
)
print(f"Dashboard: {dashboard_json['title']} ({len(dashboard_json['panels'])} panels)")
grafana_ini = """[auth.anonymous]
enabled = true
org_name = Main Org.
org_role = Viewer

[security]
allow_embedding = true

[server]
root_url = %(protocol)s://%(domain)s:%(http_port)s/
serve_from_sub_path = false
"""

grafana_ds_cm = {
    "apiVersion": "v1",
    "kind": "ConfigMap",
    "metadata": {"name": f"{GRAFANA_NAME}-datasources", "namespace": NAMESPACE},
    "data": {"datasources.yaml": datasource_provisioning},
}

grafana_dash_cm = {
    "apiVersion": "v1",
    "kind": "ConfigMap",
    "metadata": {"name": f"{GRAFANA_NAME}-dashboards-provider", "namespace": NAMESPACE},
    "data": {"dashboards.yaml": dashboard_provisioning},
}

grafana_dash_json_cm = {
    "apiVersion": "v1",
    "kind": "ConfigMap",
    "metadata": {"name": f"{GRAFANA_NAME}-dashboard-vllm", "namespace": NAMESPACE},
    "data": {"vllm-grpo-monitor.json": json.dumps(dashboard_json)},
}

grafana_ini_cm = {
    "apiVersion": "v1",
    "kind": "ConfigMap",
    "metadata": {"name": f"{GRAFANA_NAME}-ini", "namespace": NAMESPACE},
    "data": {"grafana.ini": grafana_ini},
}

grafana_deployment = {
    "apiVersion": "apps/v1",
    "kind": "Deployment",
    "metadata": {"name": GRAFANA_NAME, "namespace": NAMESPACE},
    "spec": {
        "replicas": 1,
        "selector": {"matchLabels": {"app": GRAFANA_NAME}},
        "template": {
            "metadata": {"labels": {"app": GRAFANA_NAME}},
            "spec": {
                "containers": [
                    {
                        "name": "grafana",
                        "image": "grafana/grafana:latest",
                        "ports": [{"containerPort": 3000, "name": "http"}],
                        "env": [
                            {"name": "GF_PATHS_CONFIG", "value": "/etc/grafana-custom/grafana.ini"},
                        ],
                        "resources": {
                            "requests": {"cpu": "200m", "memory": "256Mi"},
                            "limits": {"cpu": "500m", "memory": "512Mi"},
                        },
                        "volumeMounts": [
                            {"name": "datasources", "mountPath": "/etc/grafana/provisioning/datasources"},
                            {"name": "dashboards-provider", "mountPath": "/etc/grafana/provisioning/dashboards"},
                            {"name": "dashboard-vllm", "mountPath": "/var/lib/grafana/dashboards"},
                            {"name": "grafana-ini", "mountPath": "/etc/grafana-custom"},
                            {"name": "data", "mountPath": "/var/lib/grafana", "subPath": "grafana-data"},
                        ],
                    }
                ],
                "volumes": [
                    {"name": "datasources", "configMap": {"name": f"{GRAFANA_NAME}-datasources"}},
                    {"name": "dashboards-provider", "configMap": {"name": f"{GRAFANA_NAME}-dashboards-provider"}},
                    {"name": "dashboard-vllm", "configMap": {"name": f"{GRAFANA_NAME}-dashboard-vllm"}},
                    {"name": "grafana-ini", "configMap": {"name": f"{GRAFANA_NAME}-ini"}},
                    {"name": "data", "emptyDir": {}},
                ],
            },
        },
    },
}

grafana_service = {
    "apiVersion": "v1",
    "kind": "Service",
    "metadata": {"name": GRAFANA_NAME, "namespace": NAMESPACE},
    "spec": {
        "selector": {"app": GRAFANA_NAME},
        "ports": [{"port": 3000, "targetPort": 3000, "name": "http"}],
    },
}

for cm in [grafana_ds_cm, grafana_dash_cm, grafana_dash_json_cm, grafana_ini_cm]:
    _apply("ConfigMap", cm["metadata"]["name"], core.create_namespaced_config_map, core.patch_namespaced_config_map, cm)
_apply("Deployment", GRAFANA_NAME, apps.create_namespaced_deployment, apps.patch_namespaced_deployment, grafana_deployment)
_apply("Service", GRAFANA_NAME, core.create_namespaced_service, core.patch_namespaced_service, grafana_service)
print("Grafana deployed.")

## Create OpenShift Route for Grafana

In [5]:
import subprocess

route_check = subprocess.run(
    ["oc", "get", "route", GRAFANA_NAME, "-n", NAMESPACE, "-o", "jsonpath={.spec.host}"],
    capture_output=True, text=True,
)
if route_check.returncode == 0 and route_check.stdout.strip():
    grafana_host = route_check.stdout.strip()
    print(f"Route already exists: https://{grafana_host}")
else:
    subprocess.run(
        ["oc", "create", "route", "edge", GRAFANA_NAME,
         "--service", GRAFANA_NAME,
         "--port", "http",
         "-n", NAMESPACE],
        check=True,
    )
    route_out = subprocess.run(
        ["oc", "get", "route", GRAFANA_NAME, "-n", NAMESPACE, "-o", "jsonpath={.spec.host}"],
        capture_output=True, text=True, check=True,
    )
    grafana_host = route_out.stdout.strip()
    print(f"Created route: https://{grafana_host}")

GRAFANA_URL = f"https://{grafana_host}"
DASHBOARD_URL = f"{GRAFANA_URL}/d/vllm-grpo-monitor/vllm-grpo-training-monitor?orgId=1&refresh=5s"
print(f"\nGrafana URL: {GRAFANA_URL}")
print(f"Dashboard URL: {DASHBOARD_URL}")

route.route.openshift.io/vllm-grafana created
Created route: https://vllm-grafana-grpoxtrainer.apps.sridhartest-pool-7f6n4.aws.rh-ods.com

Grafana URL: https://vllm-grafana-grpoxtrainer.apps.sridhartest-pool-7f6n4.aws.rh-ods.com
Dashboard URL: https://vllm-grafana-grpoxtrainer.apps.sridhartest-pool-7f6n4.aws.rh-ods.com/d/vllm-grpo-monitor/vllm-grpo-training-monitor?orgId=1&refresh=5s


## Wait for health and verify scraping

In [6]:
prom_svc_url = f"http://{PROM_NAME}.{NAMESPACE}.svc:9090"
grafana_svc_url = f"http://{GRAFANA_NAME}.{NAMESPACE}.svc:3000"

print("Waiting for Prometheus...")
for _ in range(60):
    dep = apps.read_namespaced_deployment(name=PROM_NAME, namespace=NAMESPACE)
    ready = dep.status.ready_replicas or 0
    if ready >= 1:
        print(f"  Prometheus ready ({ready}/1)")
        break
    time.sleep(5)
else:
    raise RuntimeError("Prometheus did not become ready")

print("Waiting for Grafana...")
for _ in range(60):
    dep = apps.read_namespaced_deployment(name=GRAFANA_NAME, namespace=NAMESPACE)
    ready = dep.status.ready_replicas or 0
    if ready >= 1:
        print(f"  Grafana ready ({ready}/1)")
        break
    time.sleep(5)
else:
    raise RuntimeError("Grafana did not become ready")

time.sleep(10)

print("\nChecking Prometheus targets...")
prom_pod = core.list_namespaced_pod(namespace=NAMESPACE, label_selector=f"app={PROM_NAME}").items[0].metadata.name
targets_out = subprocess.run(
    ["oc", "exec", prom_pod, "-n", NAMESPACE, "--",
     "wget", "-qO-", "http://localhost:9090/api/v1/targets"],
    capture_output=True, text=True,
)
if targets_out.returncode == 0:
    targets = json.loads(targets_out.stdout)
    for t in targets.get("data", {}).get("activeTargets", []):
        print(f"  Target: {t['labels'].get('job', '?')} -> {t['scrapeUrl']}  health={t['health']}")
else:
    print(f"  Could not query targets: {targets_out.stderr}")

print(f"\n{'='*60}")
print(f"Grafana dashboard: {DASHBOARD_URL}")
print(f"{'='*60}")
print("\nOpen the URL above in your browser.")
print("No login needed (anonymous auth enabled).")
print("Run grpo_vllm_training.ipynb to see live metrics during training.")

Waiting for Prometheus...


  Prometheus ready (1/1)
Waiting for Grafana...


  Grafana ready (1/1)



Checking Prometheus targets...


  Target: vllm -> http://grpo-vllm-rollout.grpoxtrainer.svc:8000/metrics  health=up

Grafana dashboard: https://vllm-grafana-grpoxtrainer.apps.sridhartest-pool-7f6n4.aws.rh-ods.com/d/vllm-grpo-monitor/vllm-grpo-training-monitor?orgId=1&refresh=5s

Open the URL above in your browser.
No login needed (anonymous auth enabled).
Run grpo_vllm_training.ipynb to see live metrics during training.


## Cleanup (optional)

Run this cell to remove all Prometheus and Grafana resources.

In [7]:
CLEANUP = False  # Set to True and run to delete

if CLEANUP:
    import subprocess

    for name in [PROM_NAME, GRAFANA_NAME]:
        subprocess.run(["oc", "delete", "deployment", name, "-n", NAMESPACE, "--ignore-not-found"], check=False)
        subprocess.run(["oc", "delete", "service", name, "-n", NAMESPACE, "--ignore-not-found"], check=False)

    subprocess.run(["oc", "delete", "route", GRAFANA_NAME, "-n", NAMESPACE, "--ignore-not-found"], check=False)

    for cm_name in [
        f"{PROM_NAME}-config",
        f"{GRAFANA_NAME}-datasources",
        f"{GRAFANA_NAME}-dashboards-provider",
        f"{GRAFANA_NAME}-dashboard-vllm",
        f"{GRAFANA_NAME}-ini",
    ]:
        subprocess.run(["oc", "delete", "configmap", cm_name, "-n", NAMESPACE, "--ignore-not-found"], check=False)

    print("All Prometheus + Grafana resources deleted.")
else:
    print("Set CLEANUP = True and re-run this cell to delete resources.")

Set CLEANUP = True and re-run this cell to delete resources.
